In [8]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

In [14]:
# Revisando la extructura de las tablas scrapeadas.

from data_engine.services.scraping.bonos_locales_hacienda import obtener_tablas_crudas

tablas = obtener_tablas_crudas()
print(f"Se encontraron {len(tablas)} tablas en la página")
print(tablas[0])

Se encontraron 25 tablas en la página
           Emisiones y Reaperturas              2003                    2004  \
           Emisiones y Reaperturas      BTU-20  (UF) BTU-20  (UF) Reapertura   
0                    Fecha emisión         15-Oct-03               15-Oct-03   
1                Fecha vencimiento         15-Oct-23               15-Oct-23   
2                   Paga intereses  15-Abr /  15-Oct        15-Abr /  15-Oct   
3                   Madurez (años)                20                      20   
4  Monto Emitido Moneda Origen (1)        13.000 (2)                   7.000   
5              Tasa carátula (%a.)              4,5%                    4,5%   
6                           Status           Vigente                 Vigente   

                                  2005                                  2007  \
       BTU-20  (UF)       BTU-10  (UF)       BTU-20  (UF)        BTP-10  ($)   
0         01-Ago-04          01-Sep-05          01-Sep-05          01-Mar-07   
1

In [15]:
# revisando los encabezados de las tablas que tienen 2 niveles

print(tablas[0].columns.tolist())

[('Emisiones y Reaperturas', 'Emisiones y Reaperturas'), ('2003', 'BTU-20  (UF)'), ('2004', 'BTU-20  (UF) Reapertura'), ('2004', 'BTU-20  (UF)'), ('2005', 'BTU-10  (UF)'), ('2005', 'BTU-20  (UF)'), ('2007', 'BTP-10  ($)'), ('2007', 'BTU-20  (UF)')]


In [ ]:
# Funciones auxiliares para ordenar los encabezados.

import re
import pandas as pd


def _normalizar_nombre_campo(nombre: str) -> str:
    """Colapsa espacios múltiples y quita referencias a notas al pie
    pegadas al nombre del campo, ej. 'Monto Emitido Moneda Origen (6)' -> 'Monto Emitido Moneda Origen'."""
    nombre = re.sub(r"\s+", " ", nombre).strip()
    nombre = re.sub(r"\s*\(\d+\)\s*$", "", nombre)
    return nombre


def _limpiar_tabla_individual(tabla: pd.DataFrame) -> pd.DataFrame:
    columnas = tabla.columns.tolist()
    campo_col = columnas[0]  # ('Emisiones y Reaperturas', 'Emisiones y Reaperturas')

    tabla = tabla.copy()
    tabla[campo_col] = tabla[campo_col].apply(_normalizar_nombre_campo)
    tabla = tabla.set_index(campo_col)

    tabla_t = tabla.transpose()
    tabla_t.index = pd.MultiIndex.from_tuples(tabla_t.index, names=["anio", "nombre_bono"])
    tabla_t = tabla_t.reset_index()
    return tabla_t


def _limpiar_tablas_crudas(tablas: list[pd.DataFrame]) -> pd.DataFrame:
    tablas_limpias = [_limpiar_tabla_individual(t) for t in tablas]
    df = pd.concat(tablas_limpias, ignore_index=True)
    return df

In [17]:
# Ordenando los encabezados.

df_bonos_locales = _limpiar_tablas_crudas(tablas)
print(df_bonos_locales.shape)
print(df_bonos_locales.columns.tolist())
df_bonos_locales.head(10)

(165, 10)
['anio', 'nombre_bono', 'Fecha emisión', 'Fecha vencimiento', 'Paga intereses', 'Madurez (años)', 'Monto Emitido Moneda Origen', 'Tasa carátula (%a.)', 'Status', 'Monto Colocado']


"(Emisiones y Reaperturas, Emisiones y Reaperturas)",anio,nombre_bono,Fecha emisión,Fecha vencimiento,Paga intereses,Madurez (años),Monto Emitido Moneda Origen,Tasa carátula (%a.),Status,Monto Colocado
0,2003,BTU-20 (UF),15-Oct-03,15-Oct-23,15-Abr / 15-Oct,20,13.000 (2),"4,5%",Vigente,NaN
1,2004,BTU-20 (UF) Reapertura,15-Oct-03,15-Oct-23,15-Abr / 15-Oct,20,7.000,"4,5%",Vigente,NaN
2,2004,BTU-20 (UF),01-Ago-04,01-Ago-24,01-Feb / 01-Ago,20,18.000,"4,5%",Vigente,NaN
3,2005,BTU-10 (UF),01-Sep-05,01-Sep-15,01-Mar / 01-Sept,10,11.000,"2,1%",Vencido,NaN
4,2005,BTU-20 (UF),01-Sep-05,01-Sep-25,01-Mar / 01-Sept,20,11.000,"2,6%",Vigente,NaN
5,2007,BTP-10 ($),01-Mar-07,01-Mar-17,01-Mar / 01-Sept,10,170.000,"6,0%",Vencido,NaN
6,2007,BTU-20 (UF),01-Mar-07,01-Mar-27,01-Mar / 01-Sept,20,11.000 (3),"3,0%",Vigente,NaN
7,2008,BTP-10 ($),01-Mar-08,01-Mar-18,01-Mar/ 01-Sept,10,200.000,"6,0%",Vencido,NaN
8,2008,BTU-20 (UF),01-Mar-08,01-Mar-28,01-Mar/ 01-Sept,20,10.200,"3,0%",Vigente,NaN
9,2008,BTU-20 (UF) Reapertura,01-Mar-08,01-Mar-28,01-Mar/ 01-Sept,20,10.400 (4),"3,0%",Vigente,NaN


In [18]:
# empezando a revisar la calidad de los datos.

print(df_bonos_locales.isna().sum())
print()
print(df_bonos_locales[["anio", "nombre_bono"]].tail(20))

(Emisiones y Reaperturas, Emisiones y Reaperturas)
anio                             0
nombre_bono                      0
Fecha emisión                    0
Fecha vencimiento                0
Paga intereses                   0
Madurez (años)                 110
Monto Emitido Moneda Origen     63
Tasa carátula (%a.)              0
Status                           0
Monto Colocado                  34
dtype: int64

('Emisiones y Reaperturas', 'Emisiones y Reaperturas')  anio nombre_bono
145                                                     2024  BTU0190930
146                                                     2024  BTU0001033
147                                                     2024  BTU0300144
148                                                     2024  BTU0210750
149                                                     2024  BTU0281155
150                                                     2025  BTP0000825
151                                                     2025  BTP0001025
1

In [19]:
# Revisando si nombres repetidos de bonos son reaperturas o duplicados de datos.

print(df_bonos_locales[df_bonos_locales["nombre_bono"] == "BTP0000527"])

('Emisiones y Reaperturas', 'Emisiones y Reaperturas')  anio nombre_bono  \
136                                                     2024  BTP0000527   
156                                                     2025  BTP0000527   

('Emisiones y Reaperturas', 'Emisiones y Reaperturas') Fecha emisión  \
136                                                        01-mar-24   
156                                                        01-mar-24   

('Emisiones y Reaperturas', 'Emisiones y Reaperturas') Fecha vencimiento  \
136                                                            06-may-27   
156                                                            06-may-27   

('Emisiones y Reaperturas', 'Emisiones y Reaperturas') Paga intereses  \
136                                                           may/nov   
156                                                              n.a.   

('Emisiones y Reaperturas', 'Emisiones y Reaperturas') Madurez (años)  \
136                             

In [ ]:
# Confirmamos que, al menos en el caso de arriba, son reaperturas, ahora vamos a revisar los demás casos para ver cuantas veces se repite esto.
# Conteo de los casos.

conteo = df_bonos_locales.groupby(["nombre_bono", "Fecha emisión"]).size()
repetidos = conteo[conteo > 1]
print(f"Total de combinaciones (nombre_bono, fecha_emision) repetidas: {len(repetidos)}")
print(repetidos)

Total de combinaciones (nombre_bono, fecha_emision) repetidas: 17
nombre_bono              Fecha emisión
BTP-2035 Reap. ($)       01-mar-15        2
BTP-2043 Reap. ($)       01-ene-13        2
BTP0000527               01-mar-24        2
BTP0510750               15-ene-19        2
BTP0581029               01-abr-24        2
BTP0600433               01-abr-23        2
BTP0610456               01-abr-24        2
BTU-10 (UF)  Reapertura  01-Ene-10        2
BTU-2035 Reap. (UF)      01-mar-15        2
BTU-2044 Reap. (UF)      01-ene-14        2
BTU0001028               01-mar-21        2
BTU0190930               01-mar-18        2
BTU0210750               15-ene-19        2
BTU0281155               01-may-24        2
Letra 21M ($)            01-mar-21        2
Letra 21M ($).1          01-mar-21        2
Letra 9M ($)             01-mar-21        2
dtype: int64


In [ ]:
# Detalles de los casos

nombres_repetidos = repetidos.index.get_level_values("nombre_bono").unique()
detalle = df_bonos_locales[df_bonos_locales["nombre_bono"].isin(nombres_repetidos)].sort_values(["nombre_bono", "anio"])
print(detalle[["anio", "nombre_bono", "Fecha emisión", "Monto Colocado", "Status"]])

('Emisiones y Reaperturas', 'Emisiones y Reaperturas')  anio  \
71                                                      2017   
78                                                      2018   
72                                                      2017   
79                                                      2018   
136                                                     2024   
156                                                     2025   
126                                                     2023   
142                                                     2024   
138                                                     2024   
157                                                     2025   
125                                                     2023   
140                                                     2024   
158                                                     2025   
143                                                     2024   
160                                     

In [22]:
# Revisando las fechas de vencimiento, ayuda a diferenciar las entradas y discriminar si son duplicados o entradas distintas.

print(df_bonos_locales[df_bonos_locales["nombre_bono"].str.startswith("Letra 21M")][
    ["anio", "nombre_bono", "Fecha emisión", "Fecha vencimiento", "Monto Colocado", "Status"]
])

('Emisiones y Reaperturas', 'Emisiones y Reaperturas')  anio      nombre_bono  \
95                                                      2021    Letra 21M ($)   
101                                                     2021  Letra 21M ($).1   
103                                                     2021    Letra 21M ($)   
105                                                     2021  Letra 21M ($).1   

('Emisiones y Reaperturas', 'Emisiones y Reaperturas') Fecha emisión  \
95                                                         01-mar-21   
101                                                        01-mar-21   
103                                                        01-mar-21   
105                                                        01-mar-21   

('Emisiones y Reaperturas', 'Emisiones y Reaperturas') Fecha vencimiento  \
95                                                             21-abr-23   
101                                                            19-may-23   
103  

In [23]:
# Hay una inconsistencia en la lectura de los datos porque las fechas son tipo string y no date, 
# con esto transformamos las fechas a date y así no debiera haber filas duplicadas por ese problema

from datetime import date

MESES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dic": 12,
}

def parsear_fecha_hacienda(fecha_str: str) -> date:
    dia_str, mes_str, anio_str = fecha_str.strip().split("-")
    dia = int(dia_str)
    mes = MESES[mes_str.lower()]
    anio = 1999 if anio_str == "99" else 2000 + int(anio_str)
    return date(anio, mes, dia)

df_bonos_locales["fecha_emision_parseada"] = df_bonos_locales["Fecha emisión"].apply(parsear_fecha_hacienda)

conteo_correcto = df_bonos_locales.groupby(["nombre_bono", "fecha_emision_parseada"]).size()
repetidos_correcto = conteo_correcto[conteo_correcto > 1]
print(f"Total real de combinaciones repetidas: {len(repetidos_correcto)}")
print(repetidos_correcto)

Total real de combinaciones repetidas: 19
nombre_bono              fecha_emision_parseada
BTP-2035 Reap. ($)       2015-03-01                2
BTP-2043 Reap. ($)       2013-01-01                2
BTP0000527               2024-03-01                2
BTP0510750               2019-01-15                2
BTP0581029               2024-04-01                2
BTP0600433               2023-04-01                3
BTP0610456               2024-04-01                2
BTU-10 (UF)  Reapertura  2010-01-01                2
BTU-2035 Reap. (UF)      2015-03-01                2
BTU-2044 (UF)            2014-01-01                2
BTU-2044 Reap. (UF)      2014-01-01                2
BTU0001028               2021-03-01                2
BTU0001033               2021-03-01                2
BTU0190930               2018-03-01                2
BTU0210750               2019-01-15                2
BTU0281155               2024-05-01                2
Letra 21M ($)            2021-03-01                2
Letra 21M